##1. KPI: Total Revenue & Orders

--How much revenue are we generating and how many orders?

In [0]:
%sql
CREATE OR REPLACE VIEW ecommerce.gold.vw_kpi_revenue AS
SELECT
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(total_value) AS total_revenue,
    AVG(total_value) AS avg_order_value
FROM ecommerce.gold.fact_sales;

In [0]:
%sql
SELECT * FROM ecommerce.gold.vw_kpi_revenue;

##2. KPI: Top Selling Categories

--Which product categories drive revenue?

In [0]:
%sql
SELECT
    c.product_category_name,
    SUM(f.total_value) AS revenue
FROM ecommerce.gold.fact_sales f
JOIN ecommerce.gold.dim_category c
ON f.category_sk = c.category_sk
GROUP BY c.product_category_name
ORDER BY revenue DESC
LIMIT 10;

##3. KPI: Revenue by Location

--Which cities/states generate highest sales?

In [0]:
%sql
SELECT
    g.geolocation_state,
    SUM(f.total_value) AS revenue
FROM ecommerce.gold.fact_sales f
JOIN ecommerce.gold.dim_geolocation g
ON f.geolocation_sk = g.geolocation_sk
GROUP BY g.geolocation_state
ORDER BY revenue DESC;

##4. KPI: Delivery Performance

--How efficient is delivery?

In [0]:
%sql
SELECT
    AVG(delivery_days) AS avg_delivery_days,
    SUM(CASE WHEN delivery_delay_flag THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS delay_percentage
FROM ecommerce.gold.fact_sales;

##5. KPI: Customer Satisfaction Score

--How happy are customers?

In [0]:
%sql
SELECT
    AVG(review_score) AS avg_rating
FROM ecommerce.gold.fact_reviews;

##6. KPI: Repeat Customers

--How many customers are loyal?

In [0]:
%sql
SELECT
    COUNT(*) AS repeat_customers
FROM (
    SELECT customer_sk
    FROM ecommerce.gold.fact_sales
    GROUP BY customer_sk
    HAVING COUNT(DISTINCT order_id) > 1
);

##7. KPI: Revenue vs Rating Correlation

--Do better ratings lead to higher sales?

In [0]:
%sql
SELECT
    r.review_score,
    AVG(f.total_value) AS avg_revenue
FROM ecommerce.gold.fact_sales f
JOIN ecommerce.gold.fact_reviews r
ON f.order_id = r.order_id
GROUP BY r.review_score
ORDER BY r.review_score;

##8. KPI: Low Rating Products

--Which products are poorly rated?

In [0]:
%sql
SELECT
    p.product_category_name,
    AVG(r.review_score) AS avg_rating
FROM ecommerce.gold.fact_reviews r
JOIN ecommerce.gold.dim_product p
ON r.product_sk = p.product_sk
GROUP BY p.product_category_name
HAVING avg_rating < 3
ORDER BY avg_rating;

##9. KPI: Top Customers

--Who are your most valuable customers?

In [0]:
%sql
SELECT
    c.customer_id,
    SUM(f.total_value) AS total_spent
FROM ecommerce.gold.fact_sales f
JOIN ecommerce.gold.dim_customer c
ON f.customer_sk = c.customer_sk
GROUP BY c.customer_id
ORDER BY total_spent DESC
LIMIT 10;

##10. KPI: Delivery_delay Impacts on Ratings

In [0]:
%sql
SELECT
    f.delivery_delay_flag,
    AVG(r.review_score) AS avg_rating
FROM ecommerce.gold.fact_sales f
JOIN ecommerce.gold.fact_reviews r
ON f.order_id = r.order_id
GROUP BY f.delivery_delay_flag;

## Top Categories

In [0]:
%sql
SELECT
    c.product_category_name,
    SUM(f.total_value) AS revenue
FROM ecommerce.gold.fact_sales f
JOIN ecommerce.gold.dim_category c
ON f.category_sk = c.category_sk
GROUP BY c.product_category_name
ORDER BY revenue DESC
LIMIT 10;

Databricks visualization. Run in Databricks to view.

##1. Monthly Revenue Trend

In [0]:
%sql
SELECT
    d.year,
    d.month,
    SUM(f.total_value) AS revenue
FROM ecommerce.gold.fact_sales f
JOIN ecommerce.gold.dim_date d
ON f.date_sk = d.date_sk
GROUP BY d.year, d.month
ORDER BY d.year, d.month;

Databricks visualization. Run in Databricks to view.

##Revenue by State

In [0]:
%sql
SELECT
    g.geolocation_state,
    SUM(f.total_value) AS revenue
FROM ecommerce.gold.fact_sales f
JOIN ecommerce.gold.dim_geolocation g
ON f.geolocation_sk = g.geolocation_sk
GROUP BY g.geolocation_state;

Databricks visualization. Run in Databricks to view.

##Rating Distribution

In [0]:
%sql
SELECT
    review_score,
    COUNT(*) AS count
FROM ecommerce.gold.fact_reviews
GROUP BY review_score;

Databricks visualization. Run in Databricks to view.

##5. Delivery Delay Impact

In [0]:
%sql
SELECT
    delivery_delay_flag,
    AVG(r.review_score) AS avg_rating
FROM ecommerce.gold.fact_sales f
JOIN ecommerce.gold.fact_reviews r
ON f.order_id = r.order_id
GROUP BY delivery_delay_flag;

Databricks visualization. Run in Databricks to view.

##coverting each query to views

In [0]:
%sql 
SELECT * from ecommerce.gold.fact_sales;